# 01 — Exploratory Data Analysis

Loads `data/processed/cleaned.parquet` (produced by `make data`)
and inspects the schema, distributions, top countries, and the
daily revenue series.

**Prerequisite:** run `make pipeline` first.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams['figure.figsize'] = (10, 4)
CLEAN = Path('data/processed/cleaned.parquet')
df = pd.read_parquet(CLEAN)
print(f'Rows: {len(df):,}')
df.head()


## Quantity, UnitPrice, TotalPrice distributions

Truncate Quantity at the 99th percentile so the histogram is not
dominated by bulk-order outliers.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['Quantity', 'UnitPrice', 'TotalPrice']):
    s = df[col]
    upper = s.quantile(0.99)
    s = s[s <= upper]
    ax.hist(s, bins=40, color='steelblue', edgecolor='white')
    ax.set_title(f'{col} (≤p99 = {upper:.2f})')
    ax.set_xlabel(col)
    ax.set_ylabel('rows')
plt.tight_layout()
plt.show()


## Top 10 countries by row count

In [ ]:
top = df['Country'].value_counts().head(10)
ax = top.plot.barh(color='teal')
ax.set_xlabel('rows')
ax.set_title('Top 10 countries (rows)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()
top


## Daily revenue over the cleaned window

In [ ]:
daily = (
    df.assign(Revenue=df['Quantity'] * df['UnitPrice'])
      .set_index('InvoiceDate')
      .resample('D')['Revenue']
      .sum()
)
ax = daily.plot(color='coral', linewidth=1.2)
ax.set_title('Daily revenue (cleaned)')
ax.set_ylabel('GBP')
plt.tight_layout()
plt.show()
print(f'Days: {len(daily)}, mean: {daily.mean():.0f}, std: {daily.std():.0f}')
